# Module 04 — Exercises: The Training Loop (FashionMNIST)
FashionMNIST has the same format as MNIST (28×28 grayscale images, 10 classes) but shows clothing, and it is
harder. Write every piece of the pipeline yourself.

In [1]:
# Standard setup: make the repo root importable, pick a device, fix the random seed.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]  # notebooks live in modules/XX_name/, so go up two levels
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from common.device import get_device, seed_everything, fast_dev_run

FAST_DEV_RUN = fast_dev_run()  # True only for automated checks; leave it False when learning
DATA_DIR = ROOT / "data"
device = get_device()
seed_everything(42)

Using device: mps


In [2]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset, random_split
from torchvision import datasets, transforms

CLASSES = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]

### Exercise 1 — A custom Dataset
Write `NoisyRegression`, a `Dataset` that generates `n` points with `x ~ U(-3, 3)` and `y = sin(x) + noise`
(noise std 0.1), **once** in `__init__`. Each item should be `(x, y)` where both are float tensors of shape (1,).

In [3]:
class NoisyRegression(Dataset):
    def __init__(self, n, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.x = torch.rand(n, 1, generator=g) * 6 - 3
        self.y = torch.sin(self.x) + 0.1 * torch.randn(n, 1, generator=g)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.x[i], self.y[i]

In [4]:
ds = NoisyRegression(300)
xb, yb = next(iter(DataLoader(ds, batch_size=32)))
assert len(ds) == 300 and xb.shape == (32, 1) and yb.shape == (32, 1) and xb.dtype == torch.float32
assert xb.min() >= -3 and xb.max() <= 3
assert torch.equal(NoisyRegression(300)[5][0], ds[5][0]), "data should be generated once, deterministically"
print("Exercise 1 passed ✅")

Exercise 1 passed ✅


### Exercise 2 — Fit it with an MLP (a regression loop)
Build an `nn.Sequential` with one hidden layer of 64 units and `Tanh`, and train it with `nn.MSELoss` and
`torch.optim.Adam(lr=1e-2)` for 200 epochs over a `DataLoader(ds, batch_size=32, shuffle=True)`.
Reach a final MSE on the whole dataset below 0.02.

In [5]:
torch.manual_seed(0)
reg_model = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 1))
opt = torch.optim.Adam(reg_model.parameters(), lr=1e-2)
mse = nn.MSELoss()
loader = DataLoader(ds, batch_size=32, shuffle=True)
for epoch in range(200):
    for xb, yb in loader:
        opt.zero_grad()
        loss = mse(reg_model(xb), yb)
        loss.backward()
        opt.step()

In [6]:
with torch.no_grad():
    final = nn.functional.mse_loss(reg_model(ds.x), ds.y).item()
print("final MSE:", round(final, 4))
assert final < 0.02
print("Exercise 2 passed ✅")

final MSE: 0.0125
Exercise 2 passed ✅


### Exercise 3 — FashionMNIST loaders
Create `train_loader`, `val_loader`, `test_loader`:
- transform: `ToTensor()` then `Normalize((0.2860,), (0.3530,))`;
- split the 60k training images into 54k train / 6k validation with a seeded generator;
- batch size 128 with shuffling for train; 512 without shuffling for val/test.
(`FAST_DEV_RUN` shrinks the datasets afterwards; that code is already written.)

In [7]:
tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.2860,), (0.3530,))])
full = datasets.FashionMNIST(DATA_DIR, train=True, download=True, transform=tfm)
test_ds = datasets.FashionMNIST(DATA_DIR, train=False, download=True, transform=tfm)
train_ds, val_ds = random_split(full, [54_000, 6_000], generator=torch.Generator().manual_seed(0))
if FAST_DEV_RUN:
    train_ds, val_ds, test_ds = Subset(train_ds, range(3000)), Subset(val_ds, range(500)), Subset(test_ds, range(500))
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=512)
test_loader = DataLoader(test_ds, batch_size=512)

In [8]:
xb, yb = next(iter(train_loader))
assert xb.shape == (128, 1, 28, 28) and yb.dtype == torch.long
assert abs(xb.mean().item()) < 0.2, "did you normalize?"
if not FAST_DEV_RUN:
    assert len(train_loader.dataset) == 54_000 and len(val_loader.dataset) == 6_000
print("Exercise 3 passed ✅")

Exercise 3 passed ✅


### Exercise 4 — The model
Write `FashionMLP(nn.Module)`: flatten → Linear(784, 512) → ReLU → Linear(512, 256) → ReLU → Linear(256, 10).

In [9]:
class FashionMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.ReLU(),
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)

In [10]:
m = FashionMLP()
assert m(torch.randn(3, 1, 28, 28)).shape == (3, 10)
assert sum(p.numel() for p in m.parameters()) == 784 * 512 + 512 + 512 * 256 + 256 + 256 * 10 + 10
print("Exercise 4 passed ✅")

Exercise 4 passed ✅


### Exercise 5 — Train and evaluate loops
Write `train_epoch(model, loader, opt)` (returns the mean training loss) and `accuracy(model, loader)` (returns
the fraction correct). Use `nn.functional.cross_entropy`. Remember `train()`/`eval()`, `no_grad`, and the device.

In [11]:
def train_epoch(model, loader, opt):
    model.train()
    total, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = nn.functional.cross_entropy(model(xb), yb)
        loss.backward()
        opt.step()
        total += loss.item() * len(xb); n += len(xb)
    return total / n


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct, n = 0, 0
    for xb, yb in loader:
        pred = model(xb.to(device)).argmax(dim=1).cpu()
        correct += (pred == yb).sum().item(); n += len(yb)
    return correct / n

Now train `FashionMLP` with Adam (lr 1e-3) for 8 epochs, printing the validation accuracy each epoch.
Target: **≥ 87% test accuracy** (88–89% is typical for an MLP after 8 epochs).

In [12]:
torch.manual_seed(0)
model = FashionMLP().to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
for epoch in range(1 if FAST_DEV_RUN else 8):
    tl = train_epoch(model, train_loader, opt)
    print(f"epoch {epoch + 1}: train_loss={tl:.4f} val_acc={accuracy(model, val_loader):.4f}")

epoch 1: train_loss=1.0305 val_acc=0.8020


In [13]:
test_acc = accuracy(model, test_loader)
print("test accuracy:", round(test_acc, 4))
assert test_acc >= (0.6 if FAST_DEV_RUN else 0.87)
print("Exercise 5 passed ✅")

test accuracy: 0.762
Exercise 5 passed ✅


### Exercise 6 — Save, load and predict
Save `model`'s state_dict to `checkpoints/fashion_mlp.pt`, load it into a **new** `FashionMLP`, and write
`predict(model, imgs)` that returns a list of class **names** for a batch of images.

In [14]:
def predict(model, imgs):
    model.eval()
    with torch.no_grad():
        idx = model(imgs.to(device)).argmax(dim=1).cpu()
    return [CLASSES[i] for i in idx]


path = ROOT / "checkpoints" / "fashion_mlp.pt"
torch.save(model.state_dict(), path)
loaded = FashionMLP().to(device)
loaded.load_state_dict(torch.load(path, map_location=device))

<All keys matched successfully>

In [15]:
imgs, labels = next(iter(test_loader))
names = predict(loaded, imgs[:8])
assert names == predict(model, imgs[:8]) and all(n in CLASSES for n in names)
assert loaded is not model
print(list(zip(names, [CLASSES[i] for i in labels[:8]])))
print("Exercise 6 passed ✅")

[('Boot', 'Boot'), ('Pullover', 'Pullover'), ('Trouser', 'Trouser'), ('Trouser', 'Trouser'), ('Shirt', 'Shirt'), ('Trouser', 'Trouser'), ('Shirt', 'Coat'), ('Shirt', 'Shirt')]
Exercise 6 passed ✅
